In [1]:
# imports
import numpy as np
import pandas as pd

# plotting Libraries
import seaborn as sns
import matplotlib.pyplot as plt

# preprocessing
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer

In [2]:
# Load train data
df_train = pd.read_csv('../data/train.csv')
df_train.head(5)

,id,Time,feat1,feat2,feat3,feat4,feat5,feat6,feat7,feat8,...,feat21,feat22,feat23,feat24,feat25,feat26,feat27,feat28,Transaction_Amount,IsFraud
0,0,0.0,2.074329,-0.129425,-1.137418,0.412846,-0.192638,-1.210144,0.110697,-0.263477,...,-0.334701,-0.887840,0.336701,-0.110835,-0.291459,0.207733,-0.076576,-0.059577,1.98,0
1,1,0.0,1.998827,-1.250891,-0.520969,-0.894539,-1.122528,-0.270866,-1.029289,0.050198,...,0.054848,-0.038367,0.133518,-0.461928,-0.465491,-0.464655,-0.009413,-0.038238,84.00,0
2,2,0.0,0.091535,1.004517,-0.223445,-0.435249,0.667548,-0.988351,0.948146,-0.084789,...,-0.326725,-0.803736,0.154495,0.951233,-0.506919,0.085046,0.224458,0.087356,2.69,0
3,3,0.0,1.979649,-0.184949,-1.064206,0.120125,-0.215238,-0.648829,-0.087826,-0.035367,...,-0.095514,-0.079792,0.167701,-0.042939,0.000799,-0.096148,-0.057780,-0.073839,1.00,0
4,4,0.0,1.025898,-0.171827,1.203717,1.243900,-0.636572,1.099074,-0.938651,0.569239,...,0.099157,0.608908,0.027901,-0.262813,0.257834,-0.252829,0.108338,0.021051,1.00,0


In [3]:
# Split X and y
X_full = df_train.copy()
X_full = X_full.drop(columns=['id'])
y_full = X_full.pop('IsFraud')

In [4]:
# # Delete? I take all the features.
# # other
# from sklearn.feature_selection import mutual_info_classif

# def make_mi_scores(X,y):
#     X = X.copy()
#     for colname in X.select_dtypes(['object', 'category']):
#         x[colname], _ = X[colname].factorize()
#     # All discrete features should now have integer dtypes
#     discrete_features = [pd.api.types.is_integer_dtype(t) for t in X.dtypes]
#     mi_scores = mutual_info_classif(X,y, discrete_features = discrete_features, random_state=0)
#     mi_scores = pd.Series(mi_scores, name='MI Scores', index=X.columns)
#     mi_scores = mi_scores.sort_values(ascending=False)
#     return mi_scores

# def plot_mi_scores(scores):
#     '''Input: pandas series from def make_mi_scores'''
#     scores = scores.sort_values(ascending=True)
#     width = np.arange(len(scores))
#     ticks = list(scores.index)
#     plt.barh(width, scores)
#     plt.yticks(width, ticks)
#     plt.title('Mutual Information Scores')

# mi_scores = make_mi_scores(X_full,y_full)
# plot_mi_scores(mi_scores)

Perhaps, we can drop Time feature since it have near zero MI score.

# Data Preprocessing 

We saw in 01_eda.ipynb that all features are scaled except for the 'Time' and the 'Transaction_Amount'.
To keep same magnitute for modeling we will perform scalers for both features.
We will use RobustScaler for the 'Transaction_Amount' column as it have outliers and StandardScaler for the 'Time' column.

In [5]:
from sklearn import set_config
set_config(transform_output="pandas")

# for pipeline: scalers for the Transaction_Amount and the Time features
transformers = [
    ('robust_scale', RobustScaler(), ['Transaction_Amount']),
    ('time_scaler', StandardScaler(), ['Time'])
]

preprocessor = ColumnTransformer(
    transformers = transformers,
    remainder = 'passthrough'
)

In [6]:
X_transformed = preprocessor.fit_transform(X_full)

In [7]:
X_transformed

,robust_scale__Transaction_Amount,time_scaler__Time,remainder__feat1,remainder__feat2,remainder__feat3,remainder__feat4,remainder__feat5,remainder__feat6,remainder__feat7,remainder__feat8,...,remainder__feat19,remainder__feat20,remainder__feat21,remainder__feat22,remainder__feat23,remainder__feat24,remainder__feat25,remainder__feat26,remainder__feat27,remainder__feat28
0,-0.342039,-2.657548,2.074329,-0.129425,-1.137418,0.412846,-0.192638,-1.210144,0.110697,-0.263477,...,0.103348,-0.292969,-0.334701,-0.887840,0.336701,-0.110835,-0.291459,0.207733,-0.076576,-0.059577
1,0.881958,-2.657548,1.998827,-1.250891,-0.520969,-0.894539,-1.122528,-0.270866,-1.029289,0.050198,...,0.716784,0.065717,0.054848,-0.038367,0.133518,-0.461928,-0.465491,-0.464655,-0.009413,-0.038238
2,-0.331443,-2.657548,0.091535,1.004517,-0.223445,-0.435249,0.667548,-0.988351,0.948146,-0.084789,...,-0.433959,-0.021375,-0.326725,-0.803736,0.154495,0.951233,-0.506919,0.085046,0.224458,0.087356
3,-0.356663,-2.657548,1.979649,-0.184949,-1.064206,0.120125,-0.215238,-0.648829,-0.087826,-0.035367,...,0.642659,-0.340089,-0.095514,-0.079792,0.167701,-0.042939,0.000799,-0.096148,-0.057780,-0.073839
4,-0.356663,-2.657548,1.025898,-0.171827,1.203717,1.243900,-0.636572,1.099074,-0.938651,0.569239,...,-0.731939,-0.203628,0.099157,0.608908,0.027901,-0.262813,0.257834,-0.252829,0.108338,0.021051
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,-0.356663,1.287184,1.277125,0.665683,-0.688148,1.135626,0.494826,-0.554938,0.252478,-0.132739,...,-0.317815,-0.077668,-0.114747,-0.221548,-0.233038,-0.744995,0.799359,-0.244856,0.037868,0.042670
149996,4.847038,1.287184,0.807735,-1.813163,0.421073,-0.576839,-1.601656,-0.665517,-0.230382,-0.297095,...,0.490932,0.653768,0.270879,0.014436,-0.193669,0.387714,0.169280,-0.444501,-0.043788,0.072515
149997,1.360991,1.287184,0.836403,-0.351598,0.650338,1.066155,-0.234826,0.844271,-0.409578,0.382619,...,-1.098147,0.009924,0.285914,0.721022,-0.067894,-0.273675,0.232969,-0.286735,0.054066,0.031396
149998,2.299657,1.287184,-0.806965,0.383847,2.296469,1.428714,-2.343948,1.073324,-0.203567,0.456589,...,2.333524,0.184824,0.228740,1.296798,-0.038753,0.827484,-0.743000,0.914488,0.307576,-0.010200


In [12]:
again_df = pd.DataFrame(X_transformed, columns = X_transformed.get_feature_names_out())

AttributeError: 'numpy.ndarray' object has no attribute 'get_feature_names_out'

In [10]:
again_df

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,-0.342039,-2.657548,2.074329,-0.129425,-1.137418,0.412846,-0.192638,-1.210144,0.110697,-0.263477,...,0.103348,-0.292969,-0.334701,-0.887840,0.336701,-0.110835,-0.291459,0.207733,-0.076576,-0.059577
1,0.881958,-2.657548,1.998827,-1.250891,-0.520969,-0.894539,-1.122528,-0.270866,-1.029289,0.050198,...,0.716784,0.065717,0.054848,-0.038367,0.133518,-0.461928,-0.465491,-0.464655,-0.009413,-0.038238
2,-0.331443,-2.657548,0.091535,1.004517,-0.223445,-0.435249,0.667548,-0.988351,0.948146,-0.084789,...,-0.433959,-0.021375,-0.326725,-0.803736,0.154495,0.951233,-0.506919,0.085046,0.224458,0.087356
3,-0.356663,-2.657548,1.979649,-0.184949,-1.064206,0.120125,-0.215238,-0.648829,-0.087826,-0.035367,...,0.642659,-0.340089,-0.095514,-0.079792,0.167701,-0.042939,0.000799,-0.096148,-0.057780,-0.073839
4,-0.356663,-2.657548,1.025898,-0.171827,1.203717,1.243900,-0.636572,1.099074,-0.938651,0.569239,...,-0.731939,-0.203628,0.099157,0.608908,0.027901,-0.262813,0.257834,-0.252829,0.108338,0.021051
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,-0.356663,1.287184,1.277125,0.665683,-0.688148,1.135626,0.494826,-0.554938,0.252478,-0.132739,...,-0.317815,-0.077668,-0.114747,-0.221548,-0.233038,-0.744995,0.799359,-0.244856,0.037868,0.042670
149996,4.847038,1.287184,0.807735,-1.813163,0.421073,-0.576839,-1.601656,-0.665517,-0.230382,-0.297095,...,0.490932,0.653768,0.270879,0.014436,-0.193669,0.387714,0.169280,-0.444501,-0.043788,0.072515
149997,1.360991,1.287184,0.836403,-0.351598,0.650338,1.066155,-0.234826,0.844271,-0.409578,0.382619,...,-1.098147,0.009924,0.285914,0.721022,-0.067894,-0.273675,0.232969,-0.286735,0.054066,0.031396
149998,2.299657,1.287184,-0.806965,0.383847,2.296469,1.428714,-2.343948,1.073324,-0.203567,0.456589,...,2.333524,0.184824,0.228740,1.296798,-0.038753,0.827484,-0.743000,0.914488,0.307576,-0.010200


In [ ]:

# # Scaling Time and amount values.
# rob_scaler = RobustScaler() # is less prone to outliers

# df['scaled_Transaction_Amount'] = rob_scaler.fit_transform(df['Transaction_Amount'].values.reshape(-1,1))
# df['scaled_time'] = rob_scaler.fit_transform(df['Time'].values.reshape(-1,1))

# df.drop(['Time','Transaction_Amount'], axis=1, inplace = True)

# # we have done it, but now we need to place columns at the first and second placee in our dataframe

# scaled_amount = df['scaled_Transaction_Amount']
# scaled_time = df['scaled_time']

# df.drop(['scaled_Transaction_Amount', 'scaled_time'], axis=1, inplace=True)
# df.insert(0, 'scaled_Transaction_Amount', scaled_amount)
# df.insert(1, 'scaled_time', scaled_time)

# df.head()